In [3]:
# Load packages and required files
import folium
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import geopandas as gpd
import os
from config import *
 
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [4]:
# Sites that are in our dataset
print(MAIN_SITES)
print(f'Number of sites: {len(MAIN_SITES)}')

['STREAM-gauge-2891', 'STREAM-gauge-2886', 'STREAM-gauge-2903', 'STREAM-gauge-2962', 'STREAM-gauge-2963', 'STREAM-gauge-3092', 'STREAM-gauge-3096', 'STREAM-gauge-3097', 'STREAM-gauge-3077', 'STREAM-gauge-3089', 'STREAM-gauge-3100', 'STREAM-gauge-3108', 'STREAM-gauge-3109', 'STREAM-gauge-308', 'STREAM-gauge-2203', 'STREAM-gauge-4472', 'STREAM-gauge-4431', 'STREAM-gauge-4440', 'STREAM-gauge-4442', 'STREAM-gauge-4465', 'STREAM-gauge-2804', 'STREAM-gauge-2816', 'STREAM-gauge-3776', 'STREAM-gauge-3809', 'STREAM-gauge-4881', 'STREAM-gauge-4882', 'STREAM-gauge-695']
Number of sites: 27


In [5]:
# Read in data and filter for just oru study sites
metadata = pd.read_csv(metadata_filepath)
metadata = metadata[metadata.STREAM_ID.isin(MAIN_SITES)]
metadata.head(2)

,STREAM_ID,site_name,source,sourceID,latitude_wgs84,longitude_wgs84,state_name,time_zone,WQ_parameters,grabsamples,streamflow,COMID,catchment_shapefile_source,catchmentarea_km2,source_catchmentarea_km2,catchmentarea_variability_%,flag_catchment_shapefile,hydro_impacts
282,STREAM-gauge-308,"Mississippi River at Baton Rouge, LA",USGS,07374000,30.445667,-91.191556,Louisiana,Central,"WTemp_C,SpC_uScm,DO_mgL,pH,Turb_FNU,NO3_mgNL","SSC_mgL,NO3_mgNL,DOC_mgL,PO4_mgPL,Cl_mgL,Chla_ugL",Available,19088319.0,USGS_LID,2.880676e+06,2.915837e+06,1.205838,Safe,NaN
570,STREAM-gauge-695,"Elkhorn River at Waterloo, Nebr.",USGS,06800500,41.293333,-96.283889,Nebraska,Central,"WTemp_C,SpC_uScm,DO_mgL,Turb_FNU","SSC_mgL,TSS_mgL,NO3_mgNL,DOC_mgL,PO4_mgPL,Cl_m...",Available,24750511.0,USGS_LID,1.744614e+04,1.787093e+04,2.376989,Safe,NaN


In [7]:
# Sorting to largest watersheds are plotted first
metadata = metadata.sort_values('catchmentarea_km2', ascending=False)

# Calculate the mean latitude and longitude to center the map
mean_latitude = metadata['latitude_wgs84'].mean()
mean_longitude = metadata['longitude_wgs84'].mean()

# Create a Folium map centered at the mean coordinates
m = folium.Map(location=[mean_latitude, mean_longitude], zoom_start=4)

folium.TileLayer(
    tiles="https://tile.openstreetmap.org/{z}/{x}/{y}.png",
    attr="&copy; <a href='https://www.openstreetmap.org/copyright'>OpenStreetMap</a> contributors",
    referrer_policy="strict-origin",
).add_to(m)

# Get unique states and create a color map
unique_states = metadata['state_name'].drop_duplicates().tolist()
colors = sns.color_palette('tab10', len(unique_states)).as_hex()
state_color_map = dict(zip(unique_states, colors))

In [8]:
# Add CircleMarkers for each unique STREAM_ID, colored by state
for index, row in metadata.iterrows():
    # Isolate names
    state = row['state_name']
    stream_id = row['STREAM_ID']

    # create shapfile path
    shapefile_path = os.path.join(shapefile_filepath, f"{stream_id}.shp")
    
    color = state_color_map.get(state, '#000000') # Default to black if state not in map

    folium.CircleMarker(
        location=[row['latitude_wgs84'], row['longitude_wgs84']],
        radius=5, # Adjust size as needed
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.7,
        popup=f"Stream ID: {row['STREAM_ID']}<br>State: {row['state_name']}",
        tooltip=row['STREAM_ID']
    ).add_to(m)

    # Adding shapefiles ot map
    wtshd = gpd.read_file(shapefile_path)
    # Add to map with matching color
    folium.GeoJson(
        wtshd,
        style_function=lambda x, c=color: {
            'fillColor': c,
            'color': c,
            'weight': 2,
            'fillOpacity': 0.3
        },
        popup=f"Watershed: {stream_id}"
    ).add_to(m)            

In [9]:
# Display the map
m

In [10]:
m.save(OUTPUT_filepath+'1_analysis_files_figures/watershed_map.html')